# 🔧 Interactive Four-Bar Linkage Explorer

**For kinetic sculpture designers who think visually, not mathematically.**

Move the sliders. Watch the motion. Build intuition.

---

## Setup (Run Once)

In [ ]:
# Install required packages (run once)
!pip install matplotlib numpy ipywidgets -q
print("Ready!")

## Understanding the Four-Bar Linkage

```
    GROUND (fixed frame - the distance between your two pivot points)
    ←―――――――――――――――――――――――――――――――――――――――――――――――――――――――――――→
    
    [Fixed Pivot A]                              [Fixed Pivot B]
          ●━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━●
          ┃                                          ┃
    CRANK ┃ (input - attached to motor)      ROCKER ┃ (output - your moving element)
          ┃                                          ┃
          ●━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━●
                         COUPLER
              (the floating bar that creates interesting paths)
```

**The magic:** A point on the coupler traces a complex curve, even though you're just spinning the crank!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown, Button, HBox, VBox, Output
import warnings
warnings.filterwarnings('ignore')

# Enable interactive matplotlib
%matplotlib widget

In [ ]:
class FourBarLinkage:
    """
    A four-bar linkage solver that speaks human.
    
    ground  = distance between your two fixed pivot points
    crank   = the input arm (motor drives this)
    coupler = the floating bar (interesting motion happens here)
    rocker  = the output arm (your sculpture element attaches here)
    """
    
    def __init__(self, ground, crank, coupler, rocker):
        self.ground = ground
        self.crank = crank
        self.coupler = coupler
        self.rocker = rocker
        
        # Fixed pivot positions
        self.A = np.array([0, 0])  # Crank pivot (origin)
        self.D = np.array([ground, 0])  # Rocker pivot
        
    def get_linkage_type(self):
        """Grashof condition - determines what kind of motion is possible."""
        lengths = sorted([self.ground, self.crank, self.coupler, self.rocker])
        shortest, second, third, longest = lengths
        
        grashof_sum = shortest + longest
        other_sum = second + third
        
        if grashof_sum < other_sum:
            # Grashof linkage - at least one link can rotate fully
            if self.crank == shortest:
                return "crank-rocker", "✅ IDEAL: Crank rotates fully, rocker oscillates. Perfect for motor input!"
            elif self.ground == shortest:
                return "double-crank", "⚠️ Both crank and rocker rotate fully. Unusual but can work."
            elif self.rocker == shortest:
                return "rocker-crank", "🔄 Rocker rotates fully, crank oscillates. Swap your input/output."
            else:
                return "double-rocker", "⚠️ Neither rotates fully. Needs oscillating input, not continuous rotation."
        elif grashof_sum == other_sum:
            return "change-point", "⚠️ Special case - mechanism can flip configurations. Avoid for sculptures."
        else:
            return "non-grashof", "❌ NON-GRASHOF: No link can rotate fully. Motor can't drive this continuously."
    
    def solve_position(self, crank_angle_deg):
        """
        Given crank angle, find all joint positions.
        Returns None if position is impossible (linkage locked).
        """
        theta = np.radians(crank_angle_deg)
        
        # Point B (end of crank)
        B = self.A + self.crank * np.array([np.cos(theta), np.sin(theta)])
        
        # Now find point C (end of coupler, also end of rocker)
        # C must be:
        #   - distance 'coupler' from B
        #   - distance 'rocker' from D
        # This is a circle-circle intersection problem
        
        BD = self.D - B
        d = np.linalg.norm(BD)
        
        # Check if solution exists
        if d > self.coupler + self.rocker or d < abs(self.coupler - self.rocker):
            return None  # Circles don't intersect
        
        if d == 0 and self.coupler != self.rocker:
            return None
        
        a = (self.coupler**2 - self.rocker**2 + d**2) / (2 * d)
        h_sq = self.coupler**2 - a**2
        
        if h_sq < 0:
            return None
            
        h = np.sqrt(h_sq)
        
        # Point along BD at distance a from B
        P = B + a * BD / d
        
        # Perpendicular offset
        perp = np.array([-BD[1], BD[0]]) / d
        
        # Two solutions - we pick the "upper" one (positive y offset)
        C = P + h * perp
        
        return {
            'A': self.A,
            'B': B,
            'C': C,
            'D': self.D,
            'crank_angle': crank_angle_deg,
            'rocker_angle': np.degrees(np.arctan2(C[1] - self.D[1], C[0] - self.D[0]))
        }
    
    def get_coupler_curve(self, coupler_point_ratio=0.5, num_points=360):
        """
        Trace the path of a point on the coupler.
        
        coupler_point_ratio: 0 = at B, 1 = at C, 0.5 = middle of coupler
                            >1 or <0 = extended beyond the coupler (even more interesting paths!)
        """
        curve = []
        
        for angle in np.linspace(0, 360, num_points):
            pos = self.solve_position(angle)
            if pos is not None:
                # Interpolate point on coupler
                P = pos['B'] + coupler_point_ratio * (pos['C'] - pos['B'])
                curve.append(P)
        
        return np.array(curve) if curve else None
    
    def get_rocker_swing(self):
        """Calculate the total angular swing of the rocker."""
        angles = []
        for crank_angle in range(0, 360, 5):
            pos = self.solve_position(crank_angle)
            if pos:
                angles.append(pos['rocker_angle'])
        
        if angles:
            return max(angles) - min(angles)
        return 0

## 🎮 Interactive Explorer

**How to use:**
1. Move the sliders to change link lengths
2. Watch the linkage type indicator (green = good for motor input)
3. Observe the coupler curve (the path traced by the middle of the floating bar)
4. Adjust the coupler point slider to see different paths along the coupler

In [ ]:
# Create the interactive visualization
output = Output()

def create_linkage_plot(ground, crank, coupler, rocker, crank_angle, coupler_point, show_curve):
    """
    Create a visualization of the four-bar linkage.
    """
    linkage = FourBarLinkage(ground, crank, coupler, rocker)
    linkage_type, description = linkage.get_linkage_type()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # === LEFT PLOT: The Linkage ===
    ax1.set_aspect('equal')
    ax1.set_title(f'Four-Bar Linkage\n{description}', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    pos = linkage.solve_position(crank_angle)
    
    if pos is None:
        ax1.text(ground/2, 0, 'LOCKED!\nLinkage cannot reach this position', 
                ha='center', va='center', fontsize=14, color='red')
        ax1.set_xlim(-20, ground + 20)
        ax1.set_ylim(-ground/2, ground/2)
    else:
        A, B, C, D = pos['A'], pos['B'], pos['C'], pos['D']
        
        # Draw links
        ax1.plot([A[0], B[0]], [A[1], B[1]], 'b-', linewidth=3, label=f'Crank ({crank}mm)')
        ax1.plot([B[0], C[0]], [B[1], C[1]], 'g-', linewidth=3, label=f'Coupler ({coupler}mm)')
        ax1.plot([C[0], D[0]], [C[1], D[1]], 'r-', linewidth=3, label=f'Rocker ({rocker}mm)')
        ax1.plot([A[0], D[0]], [A[1], D[1]], 'k-', linewidth=2, label=f'Ground ({ground}mm)')
        
        # Draw joints
        for point, name, color in [(A, 'A', 'black'), (B, 'B', 'blue'), 
                                    (C, 'C', 'green'), (D, 'D', 'red')]:
            ax1.plot(point[0], point[1], 'o', markersize=10, color=color)
            ax1.annotate(name, point, xytext=(5, 5), textcoords='offset points')
        
        # Fixed pivot indicators (triangles)
        ax1.plot(A[0], A[1]-3, '^', markersize=15, color='black')
        ax1.plot(D[0], D[1]-3, '^', markersize=15, color='black')
        
        # Coupler point
        P = B + coupler_point * (C - B)
        ax1.plot(P[0], P[1], 'mo', markersize=12, label='Traced Point')
        
        # Coupler curve
        if show_curve:
            curve = linkage.get_coupler_curve(coupler_point)
            if curve is not None and len(curve) > 10:
                ax1.plot(curve[:, 0], curve[:, 1], 'm-', linewidth=1, alpha=0.5)
        
        # Set axis limits with padding
        all_points = np.array([A, B, C, D])
        if show_curve and curve is not None:
            all_points = np.vstack([all_points, curve])
        
        margin = max(ground, crank + coupler) * 0.3
        ax1.set_xlim(all_points[:, 0].min() - margin, all_points[:, 0].max() + margin)
        ax1.set_ylim(all_points[:, 1].min() - margin, all_points[:, 1].max() + margin)
        
        ax1.legend(loc='upper left', fontsize=8)
        
        # Rocker swing info
        swing = linkage.get_rocker_swing()
        ax1.text(0.02, 0.02, f'Rocker Swing: {swing:.1f}°', transform=ax1.transAxes,
                fontsize=10, verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # === RIGHT PLOT: Coupler Curve Gallery ===
    ax2.set_aspect('equal')
    ax2.set_title('Coupler Curves at Different Points', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    colors = plt.cm.viridis(np.linspace(0, 1, 5))
    for i, cp in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
        curve = linkage.get_coupler_curve(cp)
        if curve is not None and len(curve) > 10:
            ax2.plot(curve[:, 0], curve[:, 1], '-', color=colors[i], 
                    linewidth=2, alpha=0.7, label=f'Point at {cp:.0%}')
    
    ax2.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print(f"\n📊 ANALYSIS:")
    print(f"   Linkage Type: {linkage_type.upper()}")
    print(f"   {description}")
    if pos:
        print(f"   Current Rocker Angle: {pos['rocker_angle']:.1f}°")

In [ ]:
# Create interactive widget
style = {'description_width': '100px'}

interactive_plot = interactive(
    create_linkage_plot,
    ground=FloatSlider(value=100, min=20, max=200, step=5, description='Ground:', style=style),
    crank=FloatSlider(value=30, min=10, max=100, step=5, description='Crank:', style=style),
    coupler=FloatSlider(value=80, min=20, max=150, step=5, description='Coupler:', style=style),
    rocker=FloatSlider(value=70, min=20, max=150, step=5, description='Rocker:', style=style),
    crank_angle=FloatSlider(value=45, min=0, max=360, step=5, description='Crank Angle:', style=style),
    coupler_point=FloatSlider(value=0.5, min=-0.5, max=1.5, step=0.1, description='Trace Point:', style=style),
    show_curve=widgets.Checkbox(value=True, description='Show Path')
)

display(interactive_plot)

---

## 🎯 Quick Experiments to Build Intuition

Try these specific slider settings to understand what each parameter does:

### Experiment 1: The Effect of Crank Length
Keep ground=100, coupler=80, rocker=70. Change crank from 20 to 60.
- **Observation:** Shorter crank = gentler output motion

### Experiment 2: Breaking the Linkage
Set ground=100, crank=80, coupler=30, rocker=30.
- **Observation:** The linkage locks! Coupler is too short to connect.

### Experiment 3: Figure-8 Coupler Curves
Set ground=100, crank=40, coupler=100, rocker=60. Set trace point to 0.5.
- **Observation:** The coupler traces a figure-8 shape!

### Experiment 4: Extended Coupler Point
Use any working linkage. Set trace point to 1.5 (beyond the coupler).
- **Observation:** Even more dramatic curves! This is how complex automata paths are made.

---

## 🎬 Animation Generator

Once you find settings you like, generate an animation to see the full motion cycle.

In [ ]:
def animate_linkage(ground, crank, coupler, rocker, coupler_point=0.5, frames=60):
    """
    Create an animation of the linkage through one complete cycle.
    """
    linkage = FourBarLinkage(ground, crank, coupler, rocker)
    linkage_type, description = linkage.get_linkage_type()
    
    # Pre-calculate all positions
    positions = []
    coupler_curve = []
    
    for angle in np.linspace(0, 360, frames):
        pos = linkage.solve_position(angle)
        if pos:
            positions.append(pos)
            P = pos['B'] + coupler_point * (pos['C'] - pos['B'])
            coupler_curve.append(P)
    
    if not positions:
        print("Error: Linkage cannot complete a full rotation!")
        return None
    
    coupler_curve = np.array(coupler_curve)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_aspect('equal')
    ax.set_title(f'Four-Bar Animation\n{linkage_type}: {description[:50]}...', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Set fixed axis limits
    all_points = np.array([[p['A'][0], p['A'][1]] for p in positions] +
                          [[p['B'][0], p['B'][1]] for p in positions] +
                          [[p['C'][0], p['C'][1]] for p in positions] +
                          [[p['D'][0], p['D'][1]] for p in positions])
    margin = max(ground, crank + coupler) * 0.3
    ax.set_xlim(all_points[:, 0].min() - margin, all_points[:, 0].max() + margin)
    ax.set_ylim(all_points[:, 1].min() - margin, all_points[:, 1].max() + margin)
    
    # Draw static coupler curve
    ax.plot(coupler_curve[:, 0], coupler_curve[:, 1], 'm-', linewidth=1, alpha=0.3)
    
    # Initialize lines
    crank_line, = ax.plot([], [], 'b-', linewidth=3)
    coupler_line, = ax.plot([], [], 'g-', linewidth=3)
    rocker_line, = ax.plot([], [], 'r-', linewidth=3)
    ground_line, = ax.plot([], [], 'k-', linewidth=2)
    trace_point, = ax.plot([], [], 'mo', markersize=10)
    joints, = ax.plot([], [], 'ko', markersize=8)
    
    # Fixed pivots
    ax.plot(0, -3, '^', markersize=12, color='black')
    ax.plot(ground, -3, '^', markersize=12, color='black')
    
    def animate(frame):
        pos = positions[frame % len(positions)]
        A, B, C, D = pos['A'], pos['B'], pos['C'], pos['D']
        P = B + coupler_point * (C - B)
        
        crank_line.set_data([A[0], B[0]], [A[1], B[1]])
        coupler_line.set_data([B[0], C[0]], [B[1], C[1]])
        rocker_line.set_data([C[0], D[0]], [C[1], D[1]])
        ground_line.set_data([A[0], D[0]], [A[1], D[1]])
        trace_point.set_data([P[0]], [P[1]])
        joints.set_data([A[0], B[0], C[0], D[0]], [A[1], B[1], C[1], D[1]])
        
        return crank_line, coupler_line, rocker_line, ground_line, trace_point, joints
    
    anim = FuncAnimation(fig, animate, frames=len(positions), interval=50, blit=True)
    plt.close()  # Prevent static display
    
    return HTML(anim.to_jshtml())

# Example: Animate the current settings
# Modify these values to match what you found interesting in the explorer!
animate_linkage(
    ground=100,
    crank=30,
    coupler=80,
    rocker=70,
    coupler_point=0.5
)

---

## 📤 Export to OpenSCAD

Found a linkage you like? Export the coupler curve as coordinates for OpenSCAD.

In [ ]:
def export_to_openscad(ground, crank, coupler, rocker, coupler_point=0.5, filename='coupler_curve.scad'):
    """
    Export coupler curve as OpenSCAD polygon points.
    """
    linkage = FourBarLinkage(ground, crank, coupler, rocker)
    curve = linkage.get_coupler_curve(coupler_point, num_points=72)  # 5-degree steps
    
    if curve is None or len(curve) < 10:
        print("Error: Could not generate curve!")
        return
    
    # Generate OpenSCAD code
    scad_code = f"""// Four-Bar Linkage Coupler Curve
// Generated from: ground={ground}, crank={crank}, coupler={coupler}, rocker={rocker}
// Coupler point: {coupler_point}

// Linkage parameters (for reference)
GROUND = {ground};
CRANK = {crank};
COUPLER = {coupler};
ROCKER = {rocker};

// Coupler curve points
coupler_curve = [
"""
    
    for i, point in enumerate(curve):
        comma = ',' if i < len(curve) - 1 else ''
        scad_code += f"    [{point[0]:.3f}, {point[1]:.3f}]{comma}\n"
    
    scad_code += """];

// Visualization
module show_coupler_curve() {
    color("magenta", 0.5)
    linear_extrude(2)
    polygon(coupler_curve);
}

// Draw the curve
show_coupler_curve();

// Draw fixed pivots
color("black") {
    translate([0, 0, 0]) cylinder(h=5, r=3, $fn=20);
    translate([GROUND, 0, 0]) cylinder(h=5, r=3, $fn=20);
}
"""
    
    with open(filename, 'w') as f:
        f.write(scad_code)
    
    print(f"✅ Exported to {filename}")
    print(f"   Open in OpenSCAD to see the coupler curve shape.")
    print(f"   Use 'coupler_curve' array for further design.")

# Export example
export_to_openscad(100, 30, 80, 70, 0.5, 'my_linkage_curve.scad')

---

## 🧠 Key Takeaways

### The Golden Rules of Four-Bar Design

1. **Shortest link = what rotates fully**
   - Make crank the shortest → motor can spin continuously
   
2. **Crank:Rocker ratio = motion amplification**
   - Short crank + long rocker = subtle output motion (gentle sway)
   - Long crank + short rocker = dramatic output motion

3. **Coupler length controls path shape**
   - Long coupler = rounder paths
   - Short coupler = tighter, more angular paths

4. **Trace point position = path variety**
   - 0.5 (middle) = figure-8 or bean shapes
   - 0 or 1 (ends) = simpler arcs
   - Beyond 1 or below 0 = dramatic extended paths

5. **Grashof condition is non-negotiable**
   - If it says "non-grashof", your motor can't drive it continuously
   - Redesign until you see "crank-rocker" ✅